In [8]:
# 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 데이터셋 전처리 ##

In [10]:
import os
import glob
import shutil
from collections import defaultdict

# ── 경로 설정 ──────────────────────────────────────────────
base_dir     = '/content/drive/MyDrive/OnSafe/dataset'
output_dir   = '/content/drive/MyDrive/OnSafe/fms_30_dataset'
csv_root     = '/content/drive/MyDrive/OnSafe/pose_csv_results'
clean_output = '/content/drive/MyDrive/OnSafe/fms_clean_dataset'
labels       = ["ADL", "FALL"]

def process_label(label, base_dir, output_dir, csv_root, clean_output, threshold=0.5, dry_run=True):
    clip_dir = os.path.join(output_dir, label)   # 클립 위치
    orig_dir = os.path.join(base_dir, label)     # 원본 영상 위치
    csv_dir  = os.path.join(csv_root, label)     # CSV 위치

    stats = defaultdict(lambda: {"clips": 0, "csvs": 0})

    # 클립 수 집계 (fms_30_dataset/LABEL/*.mp4)
    for clip_path in glob.glob(os.path.join(clip_dir, "*.mp4")) \
                   + glob.glob(os.path.join(clip_dir, "*.MP4")):
        name = os.path.splitext(os.path.basename(clip_path))[0]  # Subject4_ADL_17_clip_02
        if "_clip_" in name:
            video_name = name.rsplit("_clip_", 1)[0]             # Subject4_ADL_17
            stats[video_name]["clips"] += 1

    # CSV 수 집계 (pose_csv_results/LABEL/*_pose.csv)
    for csv_path in glob.glob(os.path.join(csv_dir, "*_pose.csv")):
        name = os.path.splitext(os.path.basename(csv_path))[0]   # Subject4_ADL_17_clip_02_pose
        name = name.replace("_pose", "")                          # Subject4_ADL_17_clip_02
        if "_clip_" in name:
            video_name = name.rsplit("_clip_", 1)[0]             # Subject4_ADL_17
            stats[video_name]["csvs"] += 1

    # ── 판별 ──
    to_delete, to_keep = [], []

    print(f"\n{'='*65}")
    print(f"[{label}] 비디오별 클립 / CSV 현황")
    print(f"{'='*65}")

    for video_name, cnt in sorted(stats.items()):
        n_clips = cnt["clips"]
        n_csvs  = cnt["csvs"]
        ratio   = n_csvs / n_clips if n_clips > 0 else 0
        flag    = "❌ 삭제" if ratio < threshold else "✅ 유지"
        print(f"  {flag}  {video_name:40s}  클립:{n_clips:4d}  CSV:{n_csvs:4d}  ({ratio:.0%})")
        (to_delete if ratio < threshold else to_keep).append(video_name)

    print(f"\n  → 삭제 대상: {len(to_delete)}개 / 유지: {len(to_keep)}개")

    if dry_run:
        print("  ⚠️  dry_run=True → 실제 삭제는 수행하지 않았습니다.")
        return

    # ── 삭제: 원본 영상 + 클립 + CSV ──
    deleted = 0
    for video_name in to_delete:
        # 원본 영상
        for ext in ["mp4", "MP4"]:
            orig = os.path.join(orig_dir, f"{video_name}.{ext}")
            if os.path.exists(orig):
                os.remove(orig)
                deleted += 1

        # 클립
        for f in glob.glob(os.path.join(clip_dir, f"{video_name}_clip_*.mp4")):
            os.remove(f)
            deleted += 1

        # CSV
        for f in glob.glob(os.path.join(csv_dir, f"{video_name}_clip_*_pose.csv")):
            os.remove(f)
            deleted += 1

    print(f"  🗑️  삭제 완료: {deleted}개 파일 제거")

    # ── 복사: 유지 클립 + CSV → clean_output ──
    dst_clip_dir = os.path.join(clean_output, label)
    dst_csv_dir  = os.path.join(clean_output, 'pose_csv_results', label)
    os.makedirs(dst_clip_dir, exist_ok=True)
    os.makedirs(dst_csv_dir, exist_ok=True)

    copied = 0
    for video_name in to_keep:
        # 클립 복사
        for f in glob.glob(os.path.join(clip_dir, f"{video_name}_clip_*.mp4")):
            shutil.copy2(f, os.path.join(dst_clip_dir, os.path.basename(f)))
            copied += 1
        # CSV 복사
        for f in glob.glob(os.path.join(csv_dir, f"{video_name}_clip_*_pose.csv")):
            shutil.copy2(f, os.path.join(dst_csv_dir, os.path.basename(f)))
            copied += 1

    print(f"  📂 복사 완료: {copied}개 파일 → {clean_output}")


# ══════════════════════════════════════════════════════════
# Step 1: dry_run=True 로 먼저 확인
for label in labels:
    process_label(label, base_dir, output_dir, csv_root, clean_output,
                  threshold=0.5, dry_run=True)

# Step 2: 확인 후 아래 주석 해제하여 실제 실행
"""
for label in labels:
    process_label(label, base_dir, output_dir, csv_root, clean_output,
                  threshold=0.5, dry_run=False)
"""


[ADL] 비디오별 클립 / CSV 현황
  ✅ 유지  ADL_1                                     클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_10                                    클립:  20  CSV:  14  (70%)
  ✅ 유지  ADL_100                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_101                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_102                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_103                                   클립:  20  CSV:  15  (75%)
  ✅ 유지  ADL_104                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_105                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_106                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_107                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_108                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_109                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_11                                    클립:  20  CSV:  1

'\nfor label in labels:\n    process_label(label, base_dir, output_dir, csv_root, clean_output,\n                  threshold=0.5, dry_run=False)\n'

In [ ]:
process_label("ADL", base_dir, output_dir, csv_root, clean_output, threshold=0.5, dry_run=False)


[ADL] 비디오별 클립 / CSV 현황
  ✅ 유지  ADL_1                                     클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_10                                    클립:  20  CSV:  14  (70%)
  ✅ 유지  ADL_100                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_101                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_102                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_103                                   클립:  20  CSV:  15  (75%)
  ✅ 유지  ADL_104                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_105                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_106                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_107                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_108                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_109                                   클립:  20  CSV:  20  (100%)
  ✅ 유지  ADL_11                                    클립:  20  CSV:  1

In [ ]:
process_label("FALL", base_dir, output_dir, csv_root, clean_output, threshold=0.5, dry_run=False)


[FALL] 비디오별 클립 / CSV 현황
  ✅ 유지  20240912_101331                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_101427                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_101520                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_101626                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_101723                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_101943                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_102048                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_102146                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_102330                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_102649                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_102739                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_102926                           클립:   2  CSV:   2  (100%)
  ✅ 유지  20240912_103041                           클립:   2  CSV:

###파일 명 중복 제외 복사 재개 코드###

In [11]:
def process_label_resume(label, output_dir, csv_root, clean_output, threshold=0.5):
    clip_dir  = os.path.join(output_dir, label)
    csv_dir   = os.path.join(csv_root, label)
    dst_clip_dir = os.path.join(clean_output, label)
    dst_csv_dir  = os.path.join(clean_output, 'pose_csv_results', label)
    os.makedirs(dst_clip_dir, exist_ok=True)
    os.makedirs(dst_csv_dir, exist_ok=True)

    stats = defaultdict(lambda: {"clips": 0, "csvs": 0})

    for clip_path in glob.glob(os.path.join(clip_dir, "*.mp4")) \
                   + glob.glob(os.path.join(clip_dir, "*.MP4")):
        name = os.path.splitext(os.path.basename(clip_path))[0]
        if "_clip_" in name:
            stats[name.rsplit("_clip_", 1)[0]]["clips"] += 1

    for csv_path in glob.glob(os.path.join(csv_dir, "*_pose.csv")):
        name = os.path.splitext(os.path.basename(csv_path))[0].replace("_pose", "")
        if "_clip_" in name:
            stats[name.rsplit("_clip_", 1)[0]]["csvs"] += 1

    to_keep = [v for v, cnt in stats.items()
               if cnt["clips"] > 0 and cnt["csvs"] / cnt["clips"] >= threshold]

    copied_clips, copied_csvs, skipped = 0, 0, 0

    for video_name in to_keep:
        for f in glob.glob(os.path.join(clip_dir, f"{video_name}_clip_*.mp4")):
            dst = os.path.join(dst_clip_dir, os.path.basename(f))
            if not os.path.exists(dst):  # ✅ 이미 있으면 스킵
                shutil.copy2(f, dst)
                copied_clips += 1
            else:
                skipped += 1

        for f in glob.glob(os.path.join(csv_dir, f"{video_name}_clip_*_pose.csv")):
            dst = os.path.join(dst_csv_dir, os.path.basename(f))
            if not os.path.exists(dst):  # ✅ 이미 있으면 스킵
                shutil.copy2(f, dst)
                copied_csvs += 1
            else:
                skipped += 1

    print(f"[{label}] 새로 복사: 클립 {copied_clips}개 / CSV {copied_csvs}개")
    print(f"[{label}] 이미 완료: {skipped}개 스킵")

In [12]:
process_label_resume("FALL", output_dir, csv_root, clean_output, threshold=0.5)

[FALL] 새로 복사: 클립 0개 / CSV 0개
[FALL] 이미 완료: 18148개 스킵
